In [1]:
#install datases in jupyter
!pip3 install datasets pandas tqdm langchain langchain-community langchain-core python-dotenv openai langchain-openai --upgrade typing-extensions

In [2]:
import datasets 
import pandas as pd
from datasets import load_from_disk, load_dataset
import os
from tqdm.auto import tqdm
import json
datasets.logging.set_verbosity_error()

In [3]:

# dataset_r = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Electronics", trust_remote_code=True)
# dataset_m = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Electronics", trust_remote_code=True)

# dataset_r.save_to_disk('McAuley-Lab_Amazon-Reviews-2023-reviews-Electronics')
# dataset_m.save_to_disk('McAuley-Lab_Amazon-Reviews-2023-metadata-Electronics')

dataset_metadata = load_from_disk('./McAuley-Lab_Amazon-Reviews-2023-metadata-Electronics')
dataset_reviews = load_from_disk('./McAuley-Lab_Amazon-Reviews-2023-reviews-Electronics')

Loading dataset from disk:   0%|          | 0/34 [00:00<?, ?it/s]

In [4]:
# Sprawdź dostępne klucze (podzbiory) w zbiorach danych
print("Klucze w dataset_reviews:", list(dataset_reviews.keys()))
print("Klucze w dataset_metadata:", list(dataset_metadata.keys()))

Klucze w dataset_reviews: ['full']
Klucze w dataset_metadata: ['full']


In [ ]:
# Pobierz pierwszy batch aby sprawdzić strukturę danych
batch_size = 10
sample_batch = dataset_reviews['full'][:batch_size]
df_sample = pd.DataFrame(sample_batch)

# Sprawdź typy kolumn
print("Typy kolumn:")
for col in df_sample.columns:
    print(f"{col}: {type(df_sample[col].iloc[0])} - przykład: {df_sample[col].iloc[0]}")

In [ ]:
# Sprawdź strukturę metadanych
meta_sample = dataset_metadata['full'][:10]
meta_df_sample = pd.DataFrame(meta_sample)

print("Kolumny w metadanych:")
for col in meta_df_sample.columns:
    try:
        print(f"{col}: {type(meta_df_sample[col].iloc[0])} - przykład: {meta_df_sample[col].iloc[0]}")
    except:
        print(f"{col}: Nie można wyświetlić - prawdopodobnie wartość None lub złożony obiekt")

In [ ]:
# Utworzenie plików do przechowywania przetworzonych danych
reviews_output_file = 'clean_reviews.csv'
metadata_output_file = 'clean_meta.csv'

# Usuwamy pliki jeśli już istnieją
if os.path.exists(reviews_output_file):
    os.remove(reviews_output_file)
if os.path.exists(metadata_output_file):
    os.remove(metadata_output_file)

# Przetwarzanie recenzji w porcjach
print("Czyszczenie i zapisywanie recenzji...")

# Liczniki dla śledzenia przetwarzania
processed_reviews = 0
product_review_count = {}
user_review_count = {}
batch_size = 10000

# Przetwarzaj recenzje partiami
for i in tqdm(range(0, len(dataset_reviews['full']), batch_size)):
    end_idx = min(i + batch_size, len(dataset_reviews['full']))
    batch = dataset_reviews['full'][i:end_idx]
    
    df_batch = pd.DataFrame(batch)
    
    # Konwertuj kolumnę 'images' do formatu JSON string, aby uniknąć problemu z hashable
    if 'images' in df_batch.columns:
        df_batch['images'] = df_batch['images'].apply(lambda x: json.dumps(x) if isinstance(x, list) else x)
    
    # Usuwanie duplikatów w bieżącej porcji
    # Określamy podzbiór kolumn do sprawdzania duplikatów
    df_batch = df_batch.drop_duplicates(subset=['user_id', 'asin', 'rating', 'timestamp'])
    
    # Usuwanie wierszy z brakującymi wartościami
    df_batch = df_batch.dropna(subset=['user_id', 'asin', 'rating'])
    
    # Aktualizacja liczników
    for asin in df_batch['asin']:
        product_review_count[asin] = product_review_count.get(asin, 0) + 1
    
    for user_id in df_batch['user_id']:
        user_review_count[user_id] = user_review_count.get(user_id, 0) + 1
    
    # Zapisywanie przetworzonych danych do pliku
    df_batch.to_csv(reviews_output_file, mode='a', header=(i==0), index=False)
    
    processed_reviews += len(df_batch)
    
    # Wyświetlanie postępu
    if (i + batch_size) % (batch_size * 10) == 0:
        print(f"Przetworzono {processed_reviews} recenzji")

print(f"Zakończono przetwarzanie recenzji. Przetworzono {processed_reviews} wierszy.")


In [ ]:
# Teraz filtrowanie na podstawie liczby recenzji
print("Filtrowanie recenzji na podstawie liczby recenzji na produkt i użytkownika...")

# Ustalanie produktów i użytkowników do zachowania
popular_products = {asin for asin, count in product_review_count.items() if count >= 5}
active_users = {user_id for user_id, count in user_review_count.items() if count >= 2}

print(f"Po filtrowaniu: {len(popular_products)} produktów, {len(active_users)} użytkowników")

# Przetwarzanie recenzji ponownie do filtrowania
filtered_reviews_file = 'filtered_reviews.csv'
if os.path.exists(filtered_reviews_file):
    os.remove(filtered_reviews_file)

# Czytanie i filtrowanie wierszy
chunk_size = 100000  # Rozmiar porcji do czytania
filtered_count = 0

for i, chunk in enumerate(tqdm(pd.read_csv(reviews_output_file, chunksize=chunk_size))):
    # Filtrowanie na podstawie popularnych produktów i aktywnych użytkowników
    filtered_chunk = chunk[
        chunk['asin'].isin(popular_products) & 
        chunk['user_id'].isin(active_users)
    ]
    
    # Zapisywanie przefiltrowanych danych
    filtered_chunk.to_csv(filtered_reviews_file, mode='a', header=(i==0), index=False)
    filtered_count += len(filtered_chunk)

print(f"Po filtrowaniu: {filtered_count} recenzji")

In [ ]:
# Przetwarzanie metadanych produktów
print("Przetwarzanie metadanych produktów...")

# Zbiór produktów do zachowania
products_to_keep = popular_products
processed_metadata = 0

for i in tqdm(range(0, len(dataset_metadata['full']), batch_size)):
    end_idx = min(i + batch_size, len(dataset_metadata['full']))
    batch = dataset_metadata['full'][i:end_idx]
    
    df_batch = pd.DataFrame(batch)
    
    # Konwertuj kolumny zawierające listy i słowniki do formatu JSON string
    list_dict_columns = ['features', 'description', 'categories', 'images', 'videos']
    for col in list_dict_columns:
        if col in df_batch.columns:
            df_batch[col] = df_batch[col].apply(lambda x: json.dumps(x) if x is not None else None)
    
    # Usuwanie duplikatów w bieżącej porcji
    df_batch = df_batch.drop_duplicates(subset=['parent_asin'])
    
    # Filtrowanie tylko produktów, które mają wystarczająco dużo recenzji
    df_batch = df_batch[df_batch['parent_asin'].isin(products_to_keep)]
    
    # Obsługa brakujących wartości
    if 'title' in df_batch.columns:
        df_batch['title'] = df_batch['title'].fillna('')
    if 'main_category' in df_batch.columns:
        df_batch['main_category'] = df_batch['main_category'].fillna('')
    if 'store' in df_batch.columns:
        df_batch['store'] = df_batch['store'].fillna('')
    
    # Zapisywanie przetworzonych metadanych
    df_batch.to_csv(metadata_output_file, mode='a', header=(i==0), index=False)
    
    processed_metadata += len(df_batch)
    
    # Wyświetlanie postępu
    if (i + batch_size) % (batch_size * 10) == 0:
        print(f"Przetworzono {processed_metadata} produktów")

print(f"Zakończono przetwarzanie metadanych. Przetworzono {processed_metadata} produktów.")

In [5]:
# Wczytanie próbki danych do ekstrakcji cech
sample_size = 1000000  # Dostosuj w zależności od dostępnej pamięci
reviews_sample = pd.read_csv('clean_reviews.csv', nrows=sample_size)
metadata_sample = pd.read_csv('clean_meta.csv')

In [5]:
#print sample of reviews and metadata
print(reviews_sample.head())

   rating                                        title  \
0     3.0            Smells like gasoline! Going back!   
1     1.0      Didn’t work at all lenses loose/broken.   
2     5.0                                   Excellent!   
3     5.0                       Great laptop backpack!   
4     5.0  Best Headphones in the Fifties price range!   

                                                text  \
0  First & most offensive: they reek of gasoline ...   
1  These didn’t work. Idk if they were damaged in...   
2  I love these. They even come with a carry case...   
3  I was searching for a sturdy backpack for scho...   
4  I've bought these headphones three times becau...   

                                              images        asin parent_asin  \
0  [{"attachment_type": "IMAGE", "large_image_url...  B083NRGZMM  B083NRGZMM   
1                                                 []  B07N69T6TM  B07N69T6TM   
2                                                 []  B01G8JO5F2  B01G8JO5

In [6]:
print(metadata_sample.head())

               main_category  \
0            All Electronics   
1  Cell Phones & Accessories   
2          Sports & Outdoors   
3    Industrial & Scientific   
4                        NaN   

                                               title  average_rating  \
0             FS-1051 FATSHARK TELEPORTER V3 HEADSET             3.5   
1             Motorola Droid X Essentials Combo Pack             3.8   
2  Raymarine Wi-Fish DownVision Blackbox Sonar wi...             3.5   
3  Protech iPhone 4 microscope 60X Lens with illu...             3.8   
4  Fishfinder, Depth Finder Poly Sun Cover for 3"...             4.4   

   rating_number                                           features  \
0              6                                                 []   
1             64  ["New Droid X Essentials Combo Pack", "Exclusi...   
2             25  ["Black box Wi-Fi CHIRP DownVision sonar modul...   
3              6  ["iPhone 4 Microscope with White 2-LED and Det...   
4             86  [

In [7]:

print(f"Wczytano {len(reviews_sample)} przykładowych recenzji")
print(f"Wczytano {len(metadata_sample)} przykładowych metadanych produktów")

# 1. Przygotowanie metadanych produktów (proste cechy)
product_features = {}

for _, row in metadata_sample.iterrows():
    asin = row['parent_asin']
    title = row.get('title', '')
    brand = row.get('brand', '')
    
    # Połączenie cech tekstowych
    text_features = f"{title} {brand}"
    
    product_features[asin] = {
        'title': title,
        'brand': brand,
        'text_features': text_features
    }

# 2. Obliczenie podstawowych statystyk recenzji
product_stats = reviews_sample.groupby('parent_asin').agg({
    'rating': ['mean', 'count']
}).reset_index()
product_stats.columns = ['parent_asin', 'average_rating', 'num_reviews']

print("Przykładowe statystyki produktów:")
print(product_stats.head())

# Zapisanie wyników
product_stats.to_csv('product_stats_sample.csv', index=False)
print("Zapisano statystyki produktów.")

Wczytano 1000000 przykładowych recenzji
Wczytano 512465 przykładowych metadanych produktów
Przykładowe statystyki produktów:
  parent_asin  average_rating  num_reviews
0  0060897082        5.000000            1
1  0062970704        4.952381           42
2  0140194614        4.500000            2
3  0140543635        5.000000            3
4  0141188626        4.000000            1
Zapisano statystyki produktów.


In [8]:
from sklearn.feature_extraction.text import HashingVectorizer
import numpy as np

In [9]:
# Przygotowanie wektoryzatora - HashingVectorizer zamiast TfidfVectorizer
# dla wydajniejszego przetwarzania dużych zbiorów danych
vectorizer = HashingVectorizer(n_features=1000, stop_words='english')

# Inicjalizacja macierzy
product_ids = []
product_vectors = []

# Przetwarzanie partii produktów
for i, (asin, features) in enumerate(tqdm(product_features.items())):
    if i < 1000000:  # Ograniczamy do 1000000 produktów dla przykładu
        text = features['text_features']
        vector = vectorizer.transform([text]).toarray()[0]
        
        product_ids.append(asin)
        product_vectors.append(vector)

# Konwersja do DataFrame
product_vectors_df = pd.DataFrame(
    np.vstack(product_vectors),
    index=product_ids
)

print("Przykładowe wektory cech produktów:")
print(product_vectors_df.shape)

# Zapisanie wyników
product_vectors_df.to_csv('product_vectors_sample.csv')
print("Zapisano wektory cech produktów.")

  0%|          | 0/512465 [00:00<?, ?it/s]

Przykładowe wektory cech produktów:
(512465, 1000)
Zapisano wektory cech produktów.


In [7]:
product_vectors_df = pd.read_csv('product_vectors_sample.csv', index_col=0)

In [8]:
# Wczytaj przetworzone dane
reviews_df = pd.read_csv('clean_reviews.csv')
product_meta_df = pd.read_csv('clean_meta.csv')

# Połącz wektory produktów z metadanymi
product_meta_df['product_id'] = product_meta_df['parent_asin']
product_info = product_meta_df[['product_id', 'title', 'main_category', 'store', 'average_rating']].set_index('product_id')

In [9]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Funkcja do obliczania podobieństwa produktów
def get_similar_products(product_id, n=5):
    """Zwraca n produktów najbardziej podobnych do podanego product_id"""
    print(product_id)
    if product_id not in product_vectors_df.index:
        print("Product not found.")
        return []
    
    # Oblicz podobieństwo kosinusowe
    product_vector = product_vectors_df.loc[product_id].values.reshape(1, -1)
    similarities = cosine_similarity(product_vector, product_vectors_df.values)
    print("similarities", similarities)
    # Znajdź indeksy najbardziej podobnych produktów (pomijając sam produkt)
    similar_indices = similarities[0].argsort()[::-1][1:n+1]
    
    # Zwróć identyfikatory produktów
    similar_products = [product_vectors_df.index[idx] for idx in similar_indices]
    
    return similar_products

# Funkcja do generowania rekomendacji dla użytkownika
def get_recommendations_for_user(user_id, n=5):
    """Generuje rekomendacje dla użytkownika na podstawie jego historii"""
    # Znajdź produkty, które użytkownik już ocenił
    user_products = reviews_df[reviews_df['user_id'] == user_id]['asin'].tolist()
    print(user_products)
    if not user_products:
        return []
    
    # Znajdź podobne produkty dla każdego produktu, który użytkownik ocenił
    all_similar_products = []
    for product_id in user_products:
        similar_products = get_similar_products(product_id, n=3)
        all_similar_products.extend(similar_products)
    
    # Usuń duplikaty i produkty, które użytkownik już ocenił
    recommended_products = list(set(all_similar_products) - set(user_products))
    
    # Ogranicz liczbę rekomendacji
    recommended_products = recommended_products[:n]
    
    return recommended_products

# Przykład użycia
test_user_id = reviews_df['user_id'].iloc[1]
print(test_user_id)
recommendations = get_recommendations_for_user(test_user_id, n=5)

print(f"Rekomendacje dla użytkownika {test_user_id}:")
for product_id in recommendations:
    if product_id in product_info.index:
        product = product_info.loc[product_id]
        print(f"- {product['title']} (Kategoria: {product['main_category']}, Ocena: {product['average_rating']})")

AFKZENTNBQ7A7V7UXW5JJI6UGRYQ
['B083NRGZMM', 'B07N69T6TM', 'B01G8JO5F2']
B083NRGZMM
similarities [[0.         0.         0.         ... 0.         0.         0.03279129]]
B07N69T6TM
similarities [[-0.08333333  0.          0.         ...  0.          0.
   0.        ]]
B01G8JO5F2
similarities [[ 0.          0.          0.         ... -0.03175003  0.
   0.35502347]]
Rekomendacje dla użytkownika AFKZENTNBQ7A7V7UXW5JJI6UGRYQ:
- OMZER Kids Binoculars with High Resolution 8x22 Real Optic Toy Gifts for 3-9 Year Old Boys Girls, Compact Binocular for Bird Watching Travel Best Birthday Presents to 4 5 6 7 8 Years Old Child, Blue (Kategoria: Camera & Photo, Ocena: 4.2)
- SENSO Bluetooth Headphones, Best Wireless Sports Earphones w/Mic IPX7 Waterproof HD Stereo Sweatproof Earbuds for Gym Running Workout 8 Hour Battery Noise Cancelling Headsets (Kategoria: All Electronics, Ocena: 4.1)
- Binoculars, Binoculars for Adults, Binoculars for Bird Watching, Binocular, Binoculars for Hunting, Compact Binocu

In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate

os.environ["OPENAI_API_KEY"] = ""

# Inicjalizacja modelu językowego
model = init_chat_model("gpt-4o-mini", model_provider="openai")

# Szablon prompta dla rekomendacji
recommendation_prompt = PromptTemplate.from_template("""
Jesteś asystentem zakupowym, który pomaga użytkownikom znaleźć produkty elektroniczne.
Użytkownik pyta: {user_query}

Na podstawie ich zapytania, polecasz następujące produkty:
{recommendations}

Odpowiedz w formie naturalnej konwersacji, wyjaśniając dlaczego te produkty mogą być odpowiednie dla użytkownika.
Odpowiedź:
""")

recommendation_chain = recommendation_prompt | model

# Funkcja do ekstrakcji intencji z zapytania użytkownika
def extract_intent(query):
    """Analizuje zapytanie użytkownika, aby zrozumieć jego intencje zakupowe"""
    intent_prompt = PromptTemplate.from_template("""
    Przeanalizuj poniższe zapytanie użytkownika i wypisz kluczowe cechy produktu, którego szuka użytkownik:
    Zapytanie: {query}
    
    Wymień cechy w formacie JSON:
    """)
    
    intent_chain = intent_prompt | model
    
    response = intent_chain.invoke({
        "query": query
    })
    # W rzeczywistej implementacji należałoby przetworzyć odpowiedź JSON
    
    return {
        "category": "Electronics",  # Uproszczone dla przykładu
        "features": ["dobra jakość", "przystępna cena"]
    }

# Funkcja do generowania odpowiedzi na zapytanie użytkownika
def generate_response(query):
    """Generuje odpowiedź na zapytanie użytkownika wykorzystując model językowy i silnik rekomendacji"""
    # Ekstrakcja intencji
    intent = extract_intent(query)
    
    # Wyszukiwanie produktów które pasują do intencji
    # Tu należałoby zaimplementować bardziej zaawansowane filtrowanie
    sample_products = product_info.head(5)
    
    # Formatowanie rekomendacji
    recommendations_text = ""
    for idx, (prod_id, product) in enumerate(sample_products.iterrows(), 1):
        recommendations_text += f"{idx}. {product['title']} (Kategoria: {product['main_category']}, Ocena: {product['average_rating']})\n"
    
    # Generowanie odpowiedzi
    response = recommendation_chain.invoke({
        "user_query": query,
        "recommendations": recommendations_text
    })
    
    return response

# Przykładowe użycie
user_query = "Szukam dobrego aparatu cyfrowego do zdjęć przyrodniczych, najlepiej wodoodpornego."
response = generate_response(user_query)
print(response)

content='Cześć! Rozumiem, że szukasz dobrego wodoodpornego aparatu cyfrowego do robienia zdjęć przyrodniczych. Chociaż produkty, które znalazłem, nie są typowymi aparatami, mogą być dopełnieniem Twojego zestawu fotograficznego lub pomocnymi gadżetami.\n\n1. **FS-1051 FATSHARK TELEPORTER V3 HEADSET** - To headset przeznaczony głównie do dronów, które mogą być wykorzystywane w fotografii przyrodniczej. Możesz nagrywać widoki z lotu ptaka, co daje niesamowitą perspektywę. Chociaż nie jest to aparat, pozwala na uzyskanie ciekawych ujęć.\n\n2. **Motorola Droid X Essentials Combo Pack** - Zestaw ten zawiera akcesoria do telefonu, a wiele nowoczesnych smartfonów ma świetne aparaty, które są wodoodporne. Możesz rozważyć korzystanie z telefonu do zdjęć, zwłaszcza jeśli ma on odpowiednie ustawienia do robienia zdjęć w trudnych warunkach.\n\n3. **Raymarine Wi-Fish DownVision Blackbox Sonar with Wi-Fi** - To urządzenie jest bardziej związane z wędkowaniem, ale ma funkcje związane z obserwacją podw

In [ ]:
!pip3 install typing_extensions==4.7.1 --upgrade

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
# Ustawienia wizualne
plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

# Wykres rozkładu ocen
ax = sns.countplot(x='rating', data=reviews_df, palette='viridis')
plt.title('Rozkład ocen produktów', fontsize=16)
plt.xlabel('Ocena', fontsize=14)
plt.ylabel('Liczba recenzji', fontsize=14)

# Dodaj wartości liczbowe nad słupkami
for p in ax.patches:
    ax.annotate(f'{p.get_height():,}', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha = 'center', va = 'bottom', fontsize=12)

plt.tight_layout()
plt.savefig('rating_distribution.png')
plt.show()

In [ ]:
# Liczba produktów w głównych kategoriach
plt.figure(figsize=(14, 8))
category_counts = product_meta_df['main_category'].value_counts().head(15)
sns.barplot(x=category_counts.values, y=category_counts.index, palette='viridis')
plt.title('15 Najpopularniejszych kategorii produktów', fontsize=16)
plt.xlabel('Liczba produktów', fontsize=14)
plt.ylabel('Kategoria', fontsize=14)
plt.tight_layout()
plt.savefig('category_distribution.png')
plt.show()

# Średnia ocena według kategorii
# Łączenie recenzji z metadanymi
merged_df = reviews_df.merge(product_meta_df[['parent_asin', 'main_category']], 
                              left_on='asin', right_on='parent_asin', how='inner')

# Średnia ocena według kategorii
category_ratings = merged_df.groupby('main_category')['rating'].agg(['mean', 'count'])
category_ratings = category_ratings.sort_values('count', ascending=False).head(15)

plt.figure(figsize=(14, 8))
sns.barplot(x=category_ratings['mean'], y=category_ratings.index, palette='viridis')
plt.title('Średnia ocena według kategorii (15 najpopularniejszych)', fontsize=16)
plt.xlabel('Średnia ocena', fontsize=14)
plt.ylabel('Kategoria', fontsize=14)
plt.xlim(1, 5)  # Oceny są zwykle od 1 do 5
plt.tight_layout()
plt.savefig('category_ratings.png')
plt.show()

### Building item features - Content-based filtering

TF-IDF (Term Frequency-Inverse Document Frequency) is a statistical measure used in natural language processing to evaluate the importance of a word in a document relative to a collection of documents (corpus). In the context of preparing item features for a recommendation engine, TF-IDF helps transform textual data (e.g., product descriptions, reviews, or tags) into structured numerical features that capture semantic relevance, enabling the system to identify similarities between items effectively.

In [ ]:
# TF-IDF on product titles
print(formatted_metadata_df[:2]['category'])

tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=100000)
formatted_metadata_df_copy = formatted_metadata_df.copy()
title_features = tfidf_vectorizer.fit_transform(formatted_metadata_df_copy['product_title'])
title_feature_names = tfidf_vectorizer.get_feature_names_out()  # Get feature names
print(formatted_metadata_df[:2]['category'])

# print(tfidf_vectorizer.get_feature_names_out())
# print(title_features.toarray())
# One-hot encoding for categories (e.g., "Electronics > Headphones")
# formatted_metadata_df['category'] = formatted_metadata_df['category'].apply(
#     lambda x: x[0] if isinstance(x, list) else 'unknown'
# )
one_hot_encoder = OneHotEncoder(handle_unknown='ignore')
category_features = one_hot_encoder.fit_transform(formatted_metadata_df[['category']])
# category_feature_names = category_features.columns.tolist()  # Get category names

# Combine features into a sparse matrix
item_features = hstack([title_features, category_features]).tocsr()
# item_features_cols = title_feature_names.tolist() + category_feature_names

# print item features piece 

In [ ]:
from lightfm import LightFM
 
# Hybrid model using interactions + item features
model = LightFM(loss='warp', no_components=64)
model.fit(
            interactions,
            item_features=item_features,
            epochs=100,
            num_threads=4
        )

In [ ]:
print(item_features.shape)
print(one_hot_encoder.get_feature_names_out(['category']))


In [ ]:
item_features.shape

In [ ]:
category_features.shape

In [ ]:
import numpy as np

def get_recommendations(user_id: str = None, category: str = None, top_k: int = 5) -> list:
      # Cold-start handling
    if user_id is None or user_id not in user_id_map:
        scores = model.predict(0, np.arange(item_features.shape[0]))
    else:
        user_idx = user_id_map[user_id]
        scores = model.predict(user_idx, np.arange(item_features.shape[0]))
    
     # Debug scores before processing
    print(f"Initial scores stats:")
    print(f"- Range: [{scores.min():.3f}, {scores.max():.3f}]")
    print(f"- Shape: {scores.shape}")
    
    # Normalize scores to [0,1] range before masking
    scores = (scores - scores.min()) / (scores.max() - scores.min())
    print(scores[:10])

    # Filter by category
    if category:
        category_idx = get_category_feature_index(category)
        print(category_idx)
        category_mask = (item_features[:, category_idx] > 0).toarray().flatten()
        
         # Debug category mask
        print(f"\nCategory mask stats:")
        print(f"- Total items in category: {category_mask.sum()}")
        print(f"- Mask shape: {category_mask.shape}")
        
        # Get some example items before masking
        sample_indices = np.where(category_mask)[0][:3]
        if len(sample_indices) > 0:
            print("\nSample items in category before masking:")
            for idx in sample_indices:
                print(f"Score: {scores[idx]:.3f} - {merged_df.iloc[idx]['product_title']} idx: {idx}")
        
        # Apply mask
        # scores = scores * category_mask
    
    # Debug after masking
    print(f"\nAfter masking:")
    print(f"- Non-zero scores: {np.count_nonzero(scores)}")
    print(f"- Max score: {scores.max():.3f}")
    
    # Get top-k items
    available_items = merged_df['parent_asin'].unique()
    print(scores)
    valid_indices = np.where(scores > 0)[0]  # Changed from scores != 0
    print(valid_indices)
    valid_indices = valid_indices[valid_indices < len(available_items)]
    
    print(f"Scores > 0: {np.sum(scores > 0)}")
    print(f"Number of available items: {len(available_items)}")
    print(f"Valid indices shape: {valid_indices.shape}")
    
    if len(valid_indices) == 0:
        return []
    
    # Sort valid indices by scores and get top-k
    top_indices = valid_indices[np.argsort(-scores[valid_indices])[:min(top_k, len(valid_indices))]]
    # Add these lines before returning the results
    return [available_items[idx] for idx in top_indices]

def get_category_feature_index(category: str) -> int:
        # Find the column index for a given category
        try:
            return one_hot_encoder.get_feature_names_out(['category']).tolist().index(category)
        except ValueError:
            raise ValueError(f"Category '{category}' not found in item features.")

In [ ]:
print(category_feature_names[:10])
item_features.shape

get_recommendations(category='category_Car Electronics', top_k=5)

In [ ]:
# Test category existence
print("Available categories with 'Car':")
car_categories = [cat for cat in category_feature_names if 'Apple Products' in cat]
print(car_categories)

# Check items in category
category = 'category_Apple Products'
category_idx = get_category_feature_index(category)
mask = (item_features[:, category_idx] > 0).toarray().flatten()
print(f"\nTotal items in {category} [{category_idx}]: {mask.sum()}")